In [0]:
from pyspark.sql.functions import col

In [0]:
%sql
-- District-level agenda-vs-permit-activity join
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_plu_monthly_by_district AS
SELECT 
dd.district_key,
YEAR(pi.meeting_date) AS year,
MONTH(pi.meeting_date) AS month,
COUNT(*) AS total_items
FROM la_lakehouse.gold.fact_plum_items AS pi
LEFT JOIN la_lakehouse.gold.dim_district AS dd
ON pi.council_district = dd.cd
GROUP BY dd.district_key, YEAR(pi.meeting_date), MONTH(pi.meeting_date)
ORDER BY dd.district_key, YEAR(pi.meeting_date), MONTH(pi.meeting_date);

In [0]:
%sql
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_permits_monthly_by_district AS
SELECT 
    fp.district_key,
    YEAR(dd.full_date) AS year,
    MONTH(dd.full_date) AS month,
    COUNT(*) AS total_permits
FROM la_lakehouse.gold.fact_permits AS fp
LEFT JOIN la_lakehouse.gold.dim_date AS dd ON fp.submitted_date_key = dd.date_key
WHERE fp.district_key IS NOT NULL
GROUP BY fp.district_key, YEAR(dd.full_date), MONTH(dd.full_date)

In [0]:
## District-level PLU vs. Permit Activity Lag Analysis
df = spark.sql("""SELECT 
    plu.district_key,
    plu.year AS plu_year,
    plu.month AS plu_month,
    plu.total_items AS plu_items,
    permits.year AS permit_year,
    permits.month AS permit_month,
    permits.total_permits,
    -- how many months after the PLU month this permit month is
    (permits.year * 12 + permits.month) - (plu.year * 12 + plu.month) AS month_offset
FROM la_lakehouse.gold.vw_plu_monthly_by_district AS plu
JOIN la_lakehouse.gold.vw_permits_monthly_by_district AS permits
    ON plu.district_key = permits.district_key
    AND (permits.year * 12 + permits.month) - (plu.year * 12 + plu.month) BETWEEN 1 AND 3
WHERE plu.district_key IS NOT NULL
""")

In [0]:
corr_1mo = df.filter(col('month_offset') == 1).stat.corr('plu_items', 'total_permits')
corr_2mo = df.filter(col('month_offset') == 2).stat.corr('plu_items', 'total_permits')
corr_3mo = df.filter(col('month_offset') == 3).stat.corr('plu_items', 'total_permits')

print(f"1-month lag correlation: {corr_1mo:.4f}")
print(f"2-month lag correlation: {corr_2mo:.4f}")
print(f"3-month lag correlation: {corr_3mo:.4f}")

Finding: Correlations across all three lag windows (1, 2, 3 months) are negligible (-0.02 to -0.038), indicating no meaningful relationship between district-level PLUM agenda activity and subsequent permit volume. This suggests PLUM caseload (zoning appeals, CEQA reviews, policy items) and overall permit activity represent largely distinct planning processes.

In [0]:
%sql
-- Fiscal impact statement rate by item type / over time 
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_fiscal_impact_rate AS
WITH classified AS (
    SELECT 
        CASE WHEN case_no IS NOT NULL AND case_no != '' THEN 'Case-Level' ELSE 'Policy-Level' END AS item_type,
        CASE WHEN fiscal_impact_statement = 'Yes' THEN 'Yes' ELSE 'No' END AS fiscal_impact
    FROM la_lakehouse.gold.fact_plum_items
)
SELECT item_type, fiscal_impact, COUNT(*) AS total_items
FROM classified
GROUP BY item_type, fiscal_impact
ORDER BY item_type, fiscal_impact;

In [0]:
%sql
-- Case-level vs. policy-level item share
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_item_type_share AS
SELECT 
CASE WHEN case_no IS NOT NULL AND case_no != '' THEN 'Case-Level' ELSE 'Policy-Level' END AS item_type,
COUNT(*) AS total_items,
ROUND(100.0 * COUNT(*)/ SUM(COUNT(*)) OVER (), 2) AS pct_share
FROM la_lakehouse.gold.fact_plum_items
GROUP BY item_type;


In [0]:
%sql
-- How often do items get continued, and does it vary by case type?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_item_continuation AS
SELECT 
CASE WHEN case_no IS NOT NULL AND case_no != '' THEN 'Case-Level' ELSE 'Policy-Level' END AS item_type,
COUNT(*) AS total_items,
SUM(CASE WHEN is_continued THEN 1 ELSE 0 END) AS num_continued
FROM la_lakehouse.gold.fact_plum_items
GROUP BY item_type
ORDER BY total_items DESC;

In [0]:
%sql
-- Community impact submission rate over time
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_community_impact_submission_rate AS
SELECT 
YEAR(meeting_date) AS year,
CASE WHEN community_impact_submitted THEN 'Yes' ELSE 'No' END AS community_impact_submission,
COUNT(*) AS total_items,
ROUND(100.0 * COUNT(*)/ SUM(COUNT(*)) OVER (PARTITION BY YEAR(meeting_date)),2) AS submitted_share
FROM la_lakehouse.gold.fact_plum_items
GROUP BY YEAR(meeting_date), community_impact_submitted
ORDER BY year, community_impact_submitted;

#Testing Views

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_fiscal_impact_rate;

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_item_type_share;

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_item_continuation;

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_community_impact_submission_rate;

In [0]:
%sql
SELECT *
FROM la_lakehouse.gold.vw_plu_monthly_by_district;

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_permits_monthly_by_district